# Shared paths and data loaders

Common paths, study identifiers, and input readers used by the analysis notebooks.

Cells tagged `library` are loaded automatically. Run this notebook on its own to inspect the configured paths; it does not read private measurements or fit models.

## Resolve package paths

The code directory is located from the working directory or an explicit environment setting. The library loader supplies `CODE` before executing tagged cells.

In [ ]:
# Find the code folder. (When another notebook loads this library, the loader
# supplies CODE directly and this cell is not executed.)
from pathlib import Path
import os


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

## Shared paths and study constants

Supplied data default to `data/`; generated results default to `results/`. Environment variables can redirect these folders. Event and outcome order are fixed to the article.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
import numpy as np


def repository_path(value):
    """Resolve a configurable path relative to this repository, not the shell."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else CODE / path).resolve()


DATA = repository_path(os.environ.get('INDOOR_AQ_DATA', 'data'))

# Outcomes and retrofit events in manuscript order, with display labels.
OUTCOMES = ['temp', 'rh', 'pm25', 'co2']
EVENTS = ['H05_Loft', 'H05_DMEV', 'H10_Loft', 'H10_DMEV', 'H14_CWI', 'H14_Loft', 'H16_Bundle']
# Display labels use 'dMEV'; event codes such as 'H05_DMEV' are identifiers.
LABELS = dict(zip(EVENTS, ['Home 5, loft insulation', 'Home 5, dMEV', 'Home 10, loft insulation',
                          'Home 10, dMEV', 'Home 14, CWI', 'Home 14, loft insulation', 'Home 16, combined works']))
METRICS = dict(zip(OUTCOMES, ['Temperature', 'RH', 'PM2.5', 'CO2']))

## Output directory

Create the output subfolders and protect the supplied data folder.

In [ ]:
def output_root(value=None):
    """Return this run's output folder; refuse the supplied data folder."""
    path = repository_path(value or os.environ.get('INDOOR_AQ_OUTPUT', 'results'))
    protected = [DATA]
    if path == CODE or any(path == p or path.is_relative_to(p) for p in protected):
        raise ValueError('Choose a results folder separate from the supplied data folder.')
    for name in ['data', 'tables', 'figures', 'logs']:
        (path / name).mkdir(parents=True, exist_ok=True)
    return path

## Read inputs and generated results

Prepared hourly/daily inputs may be supplied in `data/` or generated in this run. Model results must exist in this run's output folder; missing steps raise an informative error.

In [ ]:
def source_file(relative, output=None):
    """Require a result produced in the selected run; no historical-result fallback."""
    path = output_root(output) / relative
    if not path.is_file():
        raise FileNotFoundError(f"Required run output is missing: {path}. Run its preceding analysis step first.")
    return path


def data_file(name, output=None):
    """Use prepared data from this run, otherwise the explicitly supplied data folder."""
    candidates = [output_root(output) / 'data' / name, DATA / name]
    for path in candidates:
        if path.is_file():
            print(f"Input data: {path}")
            return path
    raise FileNotFoundError(
        f"Missing input {name}. Prepare it with the preceding step or place an authorized copy in {DATA}. "
        "See DATA_REQUIREMENTS.md.")

## Crosswalk

Read the authoritative 23-home mapping from the private data folder. Sensor IDs are stable identifiers; historical home labels for the final three homes are not the manuscript labels.

In [ ]:
def crosswalk():
    """The 23-home crosswalk: home_id, home label, source label, sensor_id, is_treated."""
    return pd.read_csv(DATA / 'home_id_crosswalk_23.csv')

## Load hourly

Read the long-format hourly CSV, parse timestamps in UTC, and derive local dates and clock hours in Europe/London. Heating season is November through March. The optional source-label mode supports the unchanged Figure 1 mapping logic.

In [ ]:
def load_hourly(output=None, source_labels=False):
    """Hourly measurements of the 23 homes in long format.

    Adds the current home label (resolved through the sensor ID), local
    date and clock hour (Europe/London) and season (heating = Nov-Mar).
    `source_labels=True` returns the historical home labels instead; the
    Figure 1 notebook uses this to keep its original mapping logic.
    """
    x = pd.read_csv(data_file('hourly_primary_23.csv.gz', output))
    x['time'] = pd.to_datetime(x.time, utc=True)
    cw = crosswalk().set_index('sensor_id')
    # Resolve labels through sensor IDs so upstream labels cannot change the home mapping.
    x['home'] = x.sensor.map(cw['source_home' if source_labels else 'home'])
    x['home_id'] = x.sensor.map(cw.home_id)
    assert x.home.notna().all() and x.home.nunique() == 23
    x['local_time'] = x.time.dt.tz_convert('Europe/London')
    x['date'] = x.local_time.dt.strftime('%Y-%m-%d')
    x['hour'] = x.local_time.dt.hour
    x['season'] = np.where(x.local_time.dt.month.isin([11, 12, 1, 2, 3]), 'Heating season', 'Non-heating season')
    return x

## Load daily

Read daily measurements, resolve current home labels by sensor ID, and reject duplicate home-outcome-date keys. The data preparation notebook has already enforced the 18-hour minimum; this function does not fill missing outcomes.

In [ ]:
def load_daily(output=None):
    """Daily means (one row per home x outcome x local date) with season.

    Days with fewer than 18 valid hours already have y = NaN (set in
    03_prepare_cohort.ipynb); nothing is filled in here.
    """
    d = pd.read_csv(data_file('daily_home.csv', output))
    d['home'] = d.sensor_id.map(crosswalk().set_index('sensor_id').home)
    assert d.home.notna().all() and not d.duplicated(['home', 'variable', 'date']).any()
    d['season'] = np.where(pd.to_datetime(d.date).dt.month.isin([11, 12, 1, 2, 3]), 'Heating season', 'Non-heating season')
    return d

## Write a run summary

Store a UTF-8 JSON summary alongside the generated results.

In [ ]:
def dump_json(path, value):
    """Write `value` as indented UTF-8 JSON (paths and dates become strings)."""
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

## Inspect the shared configuration

This small demonstration runs only when this notebook itself is opened. The library loader skips it. Expect 23 homes and seven event identifiers.

In [ ]:
print({'homes': len(crosswalk()) if (DATA / 'home_id_crosswalk_23.csv').is_file() else 'crosswalk not supplied', 'events': len(EVENTS), 'outcomes': OUTCOMES,
       'input_data': str(DATA), 'output': str(repository_path(os.environ.get('INDOOR_AQ_OUTPUT', 'results')))})